# dectect

In [ ]:
import rclpy
from rclpy.node import Node
from sensor_msgs.msg import Image, LaserScan
from geometry_msgs.msg import Twist
from cv_bridge import CvBridge

import cv2
import numpy as np
import math

class SplitViewLaneAndObstacleNode(Node):
    def __init__(self):
        super().__init__('split_view_lane_and_obstacle')
        
        # 訂閱相機與雷達數據
        self.image_sub = self.create_subscription(
            Image, '/camera/image_raw', self.image_callback, 10)
        self.scan_sub = self.create_subscription(
            LaserScan, '/scan', self.scan_callback, 10)
        self.cmd_vel_pub = self.create_publisher(Twist, '/cmd_vel', 10)
        
        self.bridge = CvBridge()
        self.last_frame = None
        self.new_frame_available = False

        self.last_scan = None
        self.angle_min = 0.0
        self.angle_increment = 0.0

        # 定時器（約20Hz處理頻率）
        self.timer = self.create_timer(0.05, self.process_frame)

        # HSV參數 (用於線追蹤，下半部)
        self.lower_yellow = np.array([20, 100, 100])
        self.upper_yellow = np.array([30, 255, 255])
        self.lower_white  = np.array([0, 0, 180])
        self.upper_white  = np.array([255, 50, 255])
        self.area_threshold = 1000

        # 偏移量: 根據車道線，修正為路道中心 (像素)
        self.offset_yellow = 300   # 黃線通常在左，路中心 = 黃線質心 x + offset
        self.offset_white  = 300    # 白線通常在右，路中心 = 白線質心 x - offset
        
        # 為雷達數據建立障礙物極坐標視窗的參數
        self.radar_vis_size = 400  # 可視化視窗尺寸
        self.radar_scale = 80      # 將雷達讀數縮放到圖像尺寸（像素/m）
        
        # 控制參數 (下半部線追蹤控制)
        self.lane_K = 0.005  # 比例控制增益
        self.linear_speed = 0.05

        self.get_logger().info("SplitViewLaneAndObstacleNode 已啟動！")
    
    def image_callback(self, msg):
        try:
            frame = self.bridge.imgmsg_to_cv2(msg, desired_encoding='bgr8')
            self.last_frame = frame
            self.new_frame_available = True
        except Exception as e:
            self.get_logger().error("影像轉換錯誤: " + str(e))
    
    def scan_callback(self, msg):
        # 儲存最新雷達資料與角度資訊
        self.last_scan = msg.ranges
        self.angle_min = msg.angle_min
        self.angle_increment = msg.angle_increment

    def get_line_center(self, mask):
        """利用 cv2.moments 計算遮罩質心；若 m00 大於閥值則返回 (cx, cy)"""
        M = cv2.moments(mask)
        if M["m00"] > self.area_threshold:
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
            return (cx, cy)
        else:
            return None

    def get_white_line_center(self, mask):
        """
        對白色遮罩進行形態學處理與輪廓分析，過濾掉不合適的部分，
        返回候選區域中較靠下（也就是車道較前方）的白線質心。
        """
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3,3))
        mask_clean = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        contours, _ = cv2.findContours(mask_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        candidate_centers = []
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area < 200 or area > 5000:
                continue
            x, y, w_box, h_box = cv2.boundingRect(cnt)
            aspect_ratio = w_box / float(h_box)
            if aspect_ratio > 2.0:
                continue
            M = cv2.moments(cnt)
            if M["m00"] == 0:
                continue
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
            candidate_centers.append((cx, cy))
        if candidate_centers:
            return max(candidate_centers, key=lambda p: p[1])
        return None

    def process_frame(self):
        if not self.new_frame_available or self.last_frame is None:
            return

        # 複製最新整張圖
        frame = self.last_frame.copy()
        self.new_frame_available = False
        h, w, _ = frame.shape
        frame_center = w // 2

        # 先將 h//2 明確轉為 int，避免 slice 問題
        half = int(h // 2)
        # 拆分畫面：上半部 & 下半部
        top_img = frame[0:half, :]
        bottom_img = frame[half:, :]

        # ── 下半部進行車道線追蹤 ──
        hsv_bottom = cv2.cvtColor(bottom_img, cv2.COLOR_BGR2HSV)
        mask_yellow = cv2.inRange(hsv_bottom, self.lower_yellow, self.upper_yellow)
        center_lane = self.get_line_center(mask_yellow)
        lane_color = "yellow"
        if center_lane is None:
            mask_white = cv2.inRange(hsv_bottom, self.lower_white, self.upper_white)
            center_lane = self.get_white_line_center(mask_white)
            lane_color = "white"
        if center_lane is not None:
            cx_lane, cy_lane = center_lane  # ROI座標（下半部）
            cy_lane_full = cy_lane + half    # 補回全圖y座標
            if lane_color == "yellow":
                road_center_x = cx_lane + self.offset_yellow
            else:
                road_center_x = cx_lane - self.offset_white
            lane_error = road_center_x - frame_center
            lane_angular = -self.lane_K * lane_error
            cv2.circle(bottom_img, (road_center_x, cy_lane), 5, (0, 0, 255), -1)
            cv2.putText(bottom_img, f"Error: {lane_error}", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)
        else:
            lane_angular = 0.0

        # ── 上半部進行障礙物視覺檢測 ──
        # 此處將上半部轉為灰度，並根據固定閥值分割，檢查大區域（假設障礙物較大）
        gray_top = cv2.cvtColor(top_img, cv2.COLOR_BGR2GRAY)
        ret, thresh_top = cv2.threshold(gray_top, 200, 255, cv2.THRESH_BINARY)
        contours, _ = cv2.findContours(thresh_top, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        obstacle_positions = []
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area < 500:
                continue
            x, y, w_box, h_box = cv2.boundingRect(cnt)
            center_obs = (x + w_box // 2, y + h_box // 2)
            obstacle_positions.append(center_obs)
            cv2.rectangle(top_img, (x, y), (x + w_box, y + h_box), (0, 255, 0), 2)
            cv2.circle(top_img, center_obs, 5, (0, 0, 255), -1)
        cv2.imshow("Top Half - Obstacles", top_img)

        # ── 基於障礙物視覺檢測計算避障修正 ──
        # 如果檢測到障礙物，計算其平均x座標，以判斷障礙物主要集中在上半部的左邊或右邊
        obs_correction = 0.0
        if obstacle_positions:
            avg_obs_x = np.mean([pos[0] for pos in obstacle_positions])
            # 注意：top_img 的寬度與全圖的w相同
            # 若avg_obs_x < (w/2)（障礙物靠左）則向右避，給負角速度
            if avg_obs_x < (w / 2):
                obs_correction = -0.3  # 向右避
            else:
                obs_correction = 0.3   # 向左避
            self.get_logger().info(f"障礙物平均x: {avg_obs_x:.1f}, 修正: {obs_correction}")
        else:
            self.get_logger().info("無障礙物檢測到上半部。")
        
        # ── 雷達障礙物極坐標可視化 ──
        radar_vis = np.zeros((self.radar_vis_size, self.radar_vis_size, 3), np.uint8)
        center_vis = (self.radar_vis_size // 2, self.radar_vis_size // 2)
        if self.last_scan is not None:
            for i, r in enumerate(self.last_scan):
                angle = self.angle_min + i * self.angle_increment
                if math.isnan(r) or r == float('inf'):
                    continue
                x = int(center_vis[0] + self.radar_scale * r * math.cos(angle))
                y = int(center_vis[1] - self.radar_scale * r * math.sin(angle))
                cv2.circle(radar_vis, (x, y), 3, (0, 255, 255), -1)
        cv2.imshow("Obstacle Visualization (Radar)", radar_vis)

        # ── 最終控制指令組合 ──
        # 將下半部線追蹤產生的角速度與上半部的避障修正加起來得到最終角速度
        final_angular = lane_angular + obs_correction
        twist = Twist()
        # 如果下半部找不到線，則原地轉
        if center_lane is None:
            twist.linear.x = 0.0
            twist.angular.z = 0.3
        else:
            twist.linear.x = self.linear_speed
            twist.angular.z = final_angular
        self.cmd_vel_pub.publish(twist)
        self.get_logger().info(f"lane_angular: {lane_angular:.3f}, obs_correction: {obs_correction:.3f}, final: {final_angular:.3f}")

        # 在全圖上繪製分界線
        cv2.line(frame, (0, half), (w, half), (255, 0, 0), 2)
        # 將下半部處理後的圖覆蓋回全圖
        frame[half:, :] = bottom_img
        cv2.imshow("Full View with Split", frame)
        key = cv2.waitKey(1)
        if key == ord('q'):
            self.get_logger().info("使用者請求退出。")
            self.destroy_node()
            rclpy.shutdown()
            cv2.destroyAllWindows()

def main(args=None):
    rclpy.init(args=args)
    node = SplitViewLaneAndObstacleNode()
    try:
        rclpy.spin(node)
    except KeyboardInterrupt:
        node.get_logger().info("鍵盤中斷，節點關閉。")
    finally:
        node.destroy_node()
        rclpy.shutdown()
        cv2.destroyAllWindows()

if __name__ == '__main__':
    main()


# duobi dectect

In [ ]:
import rclpy
from rclpy.node import Node
from sensor_msgs.msg import Image, LaserScan
from geometry_msgs.msg import Twist
from cv_bridge import CvBridge

import cv2
import numpy as np
import math

class SplitViewLaneAndObstacleNode(Node):
    def __init__(self):
        super().__init__('split_view_lane_and_obstacle')
        
        # 訂閱相機與雷達數據
        self.image_sub = self.create_subscription(
            Image, '/camera/image_raw', self.image_callback, 10)
        self.scan_sub = self.create_subscription(
            LaserScan, '/scan', self.scan_callback, 10)
        self.cmd_vel_pub = self.create_publisher(Twist, '/cmd_vel', 10)
        
        self.bridge = CvBridge()
        self.last_frame = None
        self.new_frame_available = False

        self.last_scan = None
        self.angle_min = 0.0
        self.angle_increment = 0.0

        # 定時器（約20Hz處理頻率）
        self.timer = self.create_timer(0.05, self.process_frame)

        # HSV參數 (用於線追蹤，下半部)
        self.lower_yellow = np.array([20, 100, 100])
        self.upper_yellow = np.array([30, 255, 255])
        self.lower_white  = np.array([0, 0, 180])
        self.upper_white  = np.array([255, 50, 255])
        self.area_threshold = 1000

        # 偏移量: 根據車道線，修正為路道中心 (像素)
        self.offset_yellow = 300   # 黃線通常在左，路中心 = 黃線質心 x + offset
        self.offset_white  = 300    # 白線通常在右，路中心 = 白線質心 x - offset
        
        # 為雷達數據建立障礙物極坐標視窗的參數
        self.radar_vis_size = 400  # 可視化視窗尺寸
        self.radar_scale = 80      # 將雷達讀數縮放到圖像尺寸（像素/m）
        
        # 控制參數 (下半部線追蹤控制)
        self.lane_K = 0.005  # 比例控制增益
        self.linear_speed = 0.05

        self.get_logger().info("SplitViewLaneAndObstacleNode 已啟動！")
    
    def image_callback(self, msg):
        try:
            frame = self.bridge.imgmsg_to_cv2(msg, desired_encoding='bgr8')
            self.last_frame = frame
            self.new_frame_available = True
        except Exception as e:
            self.get_logger().error("影像轉換錯誤: " + str(e))
    
    def scan_callback(self, msg):
        # 儲存最新雷達資料與角度資訊
        self.last_scan = msg.ranges
        self.angle_min = msg.angle_min
        self.angle_increment = msg.angle_increment

    def get_line_center(self, mask):
        """利用 cv2.moments 計算遮罩質心；若 m00 大於閥值則返回 (cx, cy)"""
        M = cv2.moments(mask)
        if M["m00"] > self.area_threshold:
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
            return (cx, cy)
        else:
            return None

    def get_white_line_center(self, mask):
        """
        對白色遮罩進行形態學處理與輪廓分析，過濾掉不合適的部分，
        返回候選區域中較靠下（也就是車道較前方）的白線質心。
        """
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3,3))
        mask_clean = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        contours, _ = cv2.findContours(mask_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        candidate_centers = []
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area < 200 or area > 5000:
                continue
            x, y, w_box, h_box = cv2.boundingRect(cnt)
            aspect_ratio = w_box / float(h_box)
            if aspect_ratio > 2.0:
                continue
            M = cv2.moments(cnt)
            if M["m00"] == 0:
                continue
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
            candidate_centers.append((cx, cy))
        if candidate_centers:
            return max(candidate_centers, key=lambda p: p[1])
        return None

    def process_frame(self):
        if not self.new_frame_available or self.last_frame is None:
            return

        # 複製最新整張圖
        frame = self.last_frame.copy()
        self.new_frame_available = False
        h, w, _ = frame.shape
        frame_center = w // 2

        # 先將 h//2 明確轉為 int，避免 slice 問題
        half = int(h *0.7)
        # 拆分畫面：上半部 & 下半部
        top_img = frame[0:half, :]
        bottom_img = frame[half:, :]

        # ── 下半部進行車道線追蹤 ──
        hsv_bottom = cv2.cvtColor(bottom_img, cv2.COLOR_BGR2HSV)
        mask_yellow = cv2.inRange(hsv_bottom, self.lower_yellow, self.upper_yellow)
        center_lane = self.get_line_center(mask_yellow)
        lane_color = "yellow"
        if center_lane is None:
            mask_white = cv2.inRange(hsv_bottom, self.lower_white, self.upper_white)
            center_lane = self.get_white_line_center(mask_white)
            lane_color = "white"
        if center_lane is not None:
            cx_lane, cy_lane = center_lane  # ROI座標（下半部）
            cy_lane_full = cy_lane + half    # 補回全圖y座標
            if lane_color == "yellow":
                road_center_x = cx_lane + self.offset_yellow
            else:
                road_center_x = cx_lane - self.offset_white
            lane_error = road_center_x - frame_center
            lane_angular = -self.lane_K * lane_error
            cv2.circle(bottom_img, (road_center_x, cy_lane), 5, (0, 0, 255), -1)
            cv2.putText(bottom_img, f"Error: {lane_error}", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)
        else:
            lane_angular = 0.0

        # ── 上半部進行障礙物視覺檢測 ──
        # 此處將上半部轉為灰度，並根據固定閥值分割，檢查大區域（假設障礙物較大）
        gray_top = cv2.cvtColor(top_img, cv2.COLOR_BGR2GRAY)
        ret, thresh_top = cv2.threshold(gray_top, 200, 255, cv2.THRESH_BINARY)
        contours, _ = cv2.findContours(thresh_top, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        obstacle_positions = []
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area < 500:
                continue
            x, y, w_box, h_box = cv2.boundingRect(cnt)
            center_obs = (x + w_box // 2, y + h_box // 2)
            obstacle_positions.append(center_obs)
            cv2.rectangle(top_img, (x, y), (x + w_box, y + h_box), (0, 255, 0), 2)
            cv2.circle(top_img, center_obs, 5, (0, 0, 255), -1)
        cv2.imshow("Top Half - Obstacles", top_img)
        cv2.imshow("Bottom Half - Obstacles", bottom_img_img)

        # ── 基於障礙物視覺檢測計算避障修正 ──
        # 如果檢測到障礙物，計算其平均x座標，以判斷障礙物主要集中在上半部的左邊或右邊
        obs_correction = 0.0
        if obstacle_positions:
            avg_obs_x = np.mean([pos[0] for pos in obstacle_positions])
            # 注意：top_img 的寬度與全圖的w相同
            # 若avg_obs_x < (w/2)（障礙物靠左）則向右避，給負角速度
            if avg_obs_x < (w / 2):
                obs_correction = -0.3  # 向右避
            else:
                obs_correction = 0.3   # 向左避
            self.get_logger().info(f"障礙物平均x: {avg_obs_x:.1f}, 修正: {obs_correction}")
        else:
            self.get_logger().info("無障礙物檢測到上半部。")
        
        # ── 雷達障礙物極坐標可視化 ──
        radar_vis = np.zeros((self.radar_vis_size, self.radar_vis_size, 3), np.uint8)
        center_vis = (self.radar_vis_size // 2, self.radar_vis_size // 2)
        if self.last_scan is not None:
            for i, r in enumerate(self.last_scan):
                angle = self.angle_min + i * self.angle_increment
                if math.isnan(r) or r == float('inf'):
                    continue
                x = int(center_vis[0] + self.radar_scale * r * math.cos(angle))
                y = int(center_vis[1] - self.radar_scale * r * math.sin(angle))
                cv2.circle(radar_vis, (x, y), 3, (0, 255, 255), -1)
        cv2.imshow("Obstacle Visualization (Radar)", radar_vis)

        # ── 最終控制指令組合 ──
        # 將下半部線追蹤產生的角速度與上半部的避障修正加起來得到最終角速度
        final_angular = lane_angular + obs_correction
        twist = Twist()
        # 如果下半部找不到線，則原地轉
        if center_lane is None:
            twist.linear.x = 0.0
            twist.angular.z = 0.3
        else:
            twist.linear.x = self.linear_speed
            twist.angular.z = final_angular
        self.cmd_vel_pub.publish(twist)
        self.get_logger().info(f"lane_angular: {lane_angular:.3f}, obs_correction: {obs_correction:.3f}, final: {final_angular:.3f}")

        # 在全圖上繪製分界線
        cv2.line(frame, (0, half), (w, half), (255, 0, 0), 2)
        # 將下半部處理後的圖覆蓋回全圖
        frame[half:, :] = bottom_img
        cv2.imshow("Full View with Split", frame)
        key = cv2.waitKey(1)
        if key == ord('q'):
            self.get_logger().info("使用者請求退出。")
            self.destroy_node()
            rclpy.shutdown()
            cv2.destroyAllWindows()

def main(args=None):
    rclpy.init(args=args)
    node = SplitViewLaneAndObstacleNode()
    try:
        rclpy.spin(node)
    except KeyboardInterrupt:
        node.get_logger().info("鍵盤中斷，節點關閉。")
    finally:
        node.destroy_node()
        rclpy.shutdown()
        cv2.destroyAllWindows()

if __name__ == '__main__':
    main()


# chili

import rclpy
from rclpy.node import Node
from sensor_msgs.msg import Image, LaserScan
from geometry_msgs.msg import Twist
from cv_bridge import CvBridge

import cv2
import numpy as np
import math

class SingleLineWithPotentialField(Node):
    def __init__(self):
        super().__init__('single_line_with_potential_field')
        
        # 訂閱影像與雷達數據
        self.image_sub = self.create_subscription(
            Image, '/camera/image_raw', self.image_callback, 10)
        self.scan_sub = self.create_subscription(
            LaserScan, '/scan', self.scan_callback, 10)
        # 發布控制消息
        self.cmd_vel_pub = self.create_publisher(Twist, '/cmd_vel', 10)
        
        self.bridge = CvBridge()
        self.last_frame = None
        self.new_frame_available = False
        
        # 定時器（20 Hz）
        self.timer = self.create_timer(0.05, self.process_frame)

        # 存儲最新的雷達資料
        self.last_scan = None
        self.angle_min = 0.0
        self.angle_increment = 0.0

        # HSV參數：設定黃線和白線的HSV範圍
        self.lower_yellow = np.array([20, 100, 100])
        self.upper_yellow = np.array([30, 255, 255])
        self.lower_white = np.array([0, 0, 180])
        self.upper_white = np.array([255, 50, 255])
        self.area_threshold = 1000

        # 偏移量設定（像素）：用以修正線條位置到道路中心
        self.offset_yellow = 300   # 黃線通常在左側，所以道路中心 = 黃線中心 + offset
        self.offset_white = 300    # 白線在右側，所以道路中心 = 白線中心 - offset

        self.get_logger().info("单线结合潜在场避障版节点启动！")

    def image_callback(self, msg):
        try:
            # ROS Image轉換成OpenCV圖像
            frame = self.bridge.imgmsg_to_cv2(msg, desired_encoding='bgr8')
            self.last_frame = frame
            self.new_frame_available = True
        except Exception as e:
            self.get_logger().error("Image conversion error: " + str(e))

    def scan_callback(self, msg):
        # 存儲最新雷達資料
        self.last_scan = msg.ranges
        self.angle_min = msg.angle_min
        self.angle_increment = msg.angle_increment

    def get_line_center(self, mask):
        """
        利用cv2.moments計算遮罩內的質心。
        若m00大於預設閾值（area_threshold），返回(cx, cy)，否則返回None。
        """
        M = cv2.moments(mask)
        if M["m00"] > self.area_threshold:
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
            return (cx, cy)
        else:
            return None

    def get_white_line_center(self, mask):
        """
        對白色遮罩進行形態學處理與輪廓分析，篩選掉不符合車道線特徵的區域，
        返回候選區域中較靠下（更靠近車前）的質心。
        """
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3,3))
        mask_clean = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        contours, _ = cv2.findContours(mask_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        candidate_centers = []
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area < 200 or area > 5000:
                continue
            x, y, w_box, h_box = cv2.boundingRect(cnt)
            aspect_ratio = w_box / float(h_box)
            if aspect_ratio > 2.0:  # 車道線通常細長
                continue
            M = cv2.moments(cnt)
            if M["m00"] == 0:
                continue
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
            candidate_centers.append((cx, cy))
        if candidate_centers:
            return max(candidate_centers, key=lambda p: p[1])
        return None

    def compute_repulsive_adjustment(self):
        """
        利用最新雷達資料計算障礙物帶來的斥力修正量（潛在場方法），
        返回一個浮點值，用來調整轉向。
        算法：對所有距離小於safe_distance的雷達數據，
            累計 k_rep*(1/r - 1/safe_distance)*sin(angle) 作為橫向斥力分量。
        """
        if self.last_scan is None:
            return 0.0
        k_rep = 0.8  # 斥力常數，可調
        safe_distance = 0.7  # 安全距離（公尺）
        repulsive = 0.0
        for i, r in enumerate(self.last_scan):
            angle = self.angle_min + i * self.angle_increment
            if math.isnan(r) or r == float('inf'):
                continue
            if r < safe_distance:
                repulsive += k_rep * (1.0/r - 1.0/safe_distance) * math.sin(angle)
        return repulsive

    def process_frame(self):
        if not self.new_frame_available or self.last_frame is None:
            return
        # 複製最新全圖
        frame = self.last_frame.copy()
        self.new_frame_available = False
        h, w, _ = frame.shape
        frame_center = w // 2
        
        # ── 提取ROI：只取畫面下部40% ──
        roi_start = int(h * 0.6)
        roi = frame[roi_start:, :]
        cv2.imshow("ROI", roi)

        # 在ROI上進行HSV轉換
        hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

        # 優先嘗試黃色遮罩
        mask_yellow = cv2.inRange(hsv, self.lower_yellow, self.upper_yellow)
        cv2.imshow("Yellow Mask", mask_yellow)
        center = self.get_line_center(mask_yellow)
        detected_color = "yellow"
        # 如果黃色無效，則嘗試白色遮罩並進行輪廓過濾
        if center is None:
            mask_white = cv2.inRange(hsv, self.lower_white, self.upper_white)
            cv2.imshow("White Mask", mask_white)
            center = self.get_white_line_center(mask_white)
            detected_color = "white"
        else:
            mask_white = cv2.inRange(hsv, self.lower_white, self.upper_white)
            cv2.imshow("White Mask", mask_white)

        # 若成功提取到線條質心，計算道路中心
        if center is not None:
            cx, cy = center
            cy_full = cy + roi_start  # 將ROI內的y補回全圖
            if detected_color == "yellow":
                road_center_x = cx + self.offset_yellow
            elif detected_color == "white":
                road_center_x = cx - self.offset_white
            else:
                road_center_x = cx
            cv2.circle(frame, (road_center_x, cy_full), 5, (0, 0, 255), -1)
            error = road_center_x - frame_center
            cv2.putText(frame, f"Error: {error}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX,
                        1, (0, 255, 0), 2)
            lane_angular = -0.005 * error
        else:
            lane_angular = 0.0

        # 利用潛在場方法計算障礙物引起的斥力修正（基於雷達）
        repulsive_adjustment = self.compute_repulsive_adjustment()
        # 注意：如果障礙物在左側，sin(angle)正，repulsive_adjustment會為正，
        # 此時希望向右（負角速度）修正，因此最終角速度作差：
        final_angular = lane_angular - repulsive_adjustment

        twist = Twist()
        twist.linear.x = 0.05
        twist.angular.z = final_angular
        self.cmd_vel_pub.publish(twist)
        self.get_logger().info(f"lane: {lane_angular:.3f}, repulsive: {repulsive_adjustment:.3f}, final: {final_angular:.3f}")
        
        cv2.imshow("Line Tracking", frame)
        key = cv2.waitKey(1)
        if key == ord('q'):
            self.get_logger().info("User requested exit.")
            self.destroy_node()
            rclpy.shutdown()
            cv2.destroyAllWindows()

def main(args=None):
    rclpy.init(args=args)
    node = SingleLineWithPotentialField()
    try:
        rclpy.spin(node)
    except KeyboardInterrupt:
        node.get_logger().info("Keyboard interrupt, shutting down.")
    finally:
        node.destroy_node()
        rclpy.shutdown()
        cv2.destroyAllWindows()

if __name__ == '__main__':
    main()
